# Исследование надежности заемщиков


## Откройте таблицу и изучите общую информацию о данных

**Задание 1. Импортируйте библиотеку pandas. Считайте данные из csv-файла в датафрейм и сохраните в переменную `data`. Путь к файлу:**

`/datasets/data.csv`

In [1]:
import pandas as pd

try:
    data = pd.read_csv('/datasets/data.csv')
except:
    data = pd.read_csv('https://code.s3.yandex.net/datasets/data.csv')

**Задание 2. Выведите первые 20 строчек датафрейма `data` на экран.**

In [2]:
data.head(20)

,children,days_employed,dob_years,education,education_id,family_status,family_status_id,gender,income_type,debt,total_income,purpose
0,1,-8437.673028,42,высшее,0,женат / замужем,0,F,сотрудник,0,253875.639453,покупка жилья
1,1,-4024.803754,36,среднее,1,женат / замужем,0,F,сотрудник,0,112080.014102,приобретение автомобиля
2,0,-5623.422610,33,Среднее,1,женат / замужем,0,M,сотрудник,0,145885.952297,покупка жилья
3,3,-4124.747207,32,среднее,1,женат / замужем,0,M,сотрудник,0,267628.550329,дополнительное образование
4,0,340266.072047,53,среднее,1,гражданский брак,1,F,пенсионер,0,158616.077870,сыграть свадьбу
5,0,-926.185831,27,высшее,0,гражданский брак,1,M,компаньон,0,255763.565419,покупка жилья
6,0,-2879.202052,43,высшее,0,женат / замужем,0,F,компаньон,0,240525.971920,операции с жильем
7,0,-152.779569,50,СРЕДНЕЕ,1,женат / замужем,0,M,сотрудник,0,135823.934197,образование
8,2,-6929.865299,35,ВЫСШЕЕ,0,гражданский брак,1,F,сотрудник,0,95856.832424,на проведение свадьбы
9,0,-2188.756445,41,среднее,1,женат / замужем,0,M,сотрудник,0,144425.938277,покупка жилья для семьи


**Задание 3. Выведите основную информацию о датафрейме с помощью метода `info()`.**

In [3]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 21525 entries, 0 to 21524
Data columns (total 12 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   children          21525 non-null  int64  
 1   days_employed     19351 non-null  float64
 2   dob_years         21525 non-null  int64  
 3   education         21525 non-null  object 
 4   education_id      21525 non-null  int64  
 5   family_status     21525 non-null  object 
 6   family_status_id  21525 non-null  int64  
 7   gender            21525 non-null  object 
 8   income_type       21525 non-null  object 
 9   debt              21525 non-null  int64  
 10  total_income      19351 non-null  float64
 11  purpose           21525 non-null  object 
dtypes: float64(2), int64(5), object(5)
memory usage: 2.0+ MB


## Предобработка данных

### Удаление пропусков

**Задание 4. Выведите количество пропущенных значений для каждого столбца. Используйте комбинацию двух методов.**

In [4]:
data.isna().sum()

children               0
days_employed       2174
dob_years              0
education              0
education_id           0
family_status          0
family_status_id       0
gender                 0
income_type            0
debt                   0
total_income        2174
purpose                0
dtype: int64

**Задание 5. В двух столбцах есть пропущенные значения. Один из них — `days_employed`. Пропуски в этом столбце вы обработаете на следующем этапе. Другой столбец с пропущенными значениями — `total_income` — хранит данные о доходах. На сумму дохода сильнее всего влияет тип занятости, поэтому заполнить пропуски в этом столбце нужно медианным значением по каждому типу из столбца `income_type`. Например, у человека с типом занятости `сотрудник` пропуск в столбце `total_income` должен быть заполнен медианным доходом среди всех записей с тем же типом.**

In [5]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['total_income'].isna()), 'total_income'] = \
    data.loc[(data['income_type'] == t), 'total_income'].median()

### Обработка аномальных значений

**Задание 6. В данных могут встречаться артефакты (аномалии) — значения, которые не отражают действительность и появились по какой-то ошибке. таким артефактом будет отрицательное количество дней трудового стажа в столбце `days_employed`. Для реальных данных это нормально. Обработайте значения в этом столбце: замените все отрицательные значения положительными с помощью метода `abs()`.**

In [6]:
data['days_employed'] = data['days_employed'].abs()

**Задание 7. Для каждого типа занятости выведите медианное значение трудового стажа `days_employed` в днях.**

In [7]:
data.groupby('income_type')['days_employed'].agg('median')

income_type
безработный        366413.652744
в декрете            3296.759962
госслужащий          2689.368353
компаньон            1547.382223
пенсионер          365213.306266
предприниматель       520.848083
сотрудник            1574.202821
студент               578.751554
Name: days_employed, dtype: float64

У двух типов (безработные и пенсионеры) получатся аномально большие значения. Исправить такие значения сложно, поэтому оставьте их как есть. Тем более этот столбец не понадобится вам для исследования.

**Задание 8. Выведите перечень уникальных значений столбца `children`.**

In [8]:
data['children'].unique()

array([ 1,  0,  3,  2, -1,  4, 20,  5])

**Задание 9. В столбце `children` есть два аномальных значения. Удалите строки, в которых встречаются такие аномальные значения из датафрейма `data`.**

In [9]:
data = data[(data['children'] != -1) & (data['children'] != 20)]

**Задание 10. Ещё раз выведите перечень уникальных значений столбца `children`, чтобы убедиться, что артефакты удалены.**

In [10]:
data['children'].unique()

array([1, 0, 3, 2, 4, 5])

### Удаление пропусков (продолжение)

**Задание 11. Заполните пропуски в столбце `days_employed` медианными значениями по каждого типа занятости `income_type`.**

In [11]:
for t in data['income_type'].unique():
    data.loc[(data['income_type'] == t) & (data['days_employed'].isna()), 'days_employed'] = \
    data.loc[(data['income_type'] == t), 'days_employed'].median()

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

**Задание 12. Убедитесь, что все пропуски заполнены. Проверьте себя и ещё раз выведите количество пропущенных значений для каждого столбца с помощью двух методов.**

In [12]:
data.isna().sum()

children            0
days_employed       0
dob_years           0
education           0
education_id        0
family_status       0
family_status_id    0
gender              0
income_type         0
debt                0
total_income        0
purpose             0
dtype: int64

### Изменение типов данных

**Задание 13. Замените вещественный тип данных в столбце `total_income` на целочисленный с помощью метода `astype()`.**

In [13]:
data['total_income'] = data['total_income'].astype(int)

### Обработка дубликатов

**Задание 14. Обработайте неявные дубликаты в столбце `education`. В этом столбце есть одни и те же значения, но записанные по-разному: с использованием заглавных и строчных букв. Приведите их к нижнему регистру. Проверьте остальные столбцы.**

In [14]:
data['education'] = data['education'].str.lower()

**Задание 15. Выведите на экран количество строк-дубликатов в данных. Если такие строки присутствуют, удалите их.**

In [15]:
data.duplicated().sum()

71

In [16]:
data = data.drop_duplicates()

### Категоризация данных

**Задание 16. На основании диапазонов, указанных ниже, создайте в датафрейме `data` столбец `total_income_category` с категориями:**

- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.


**Например, кредитополучателю с доходом 25000 нужно назначить категорию `'E'`, а клиенту, получающему 235000, — `'B'`. Используйте собственную функцию с именем `categorize_income()` и метод `apply()`.**

In [17]:
def categorize_income(income):
    try:
        if 0 <= income <= 30000:
            return 'E'
        elif 30001 <= income <= 50000:
            return 'D'
        elif 50001 <= income <= 200000:
            return 'C'
        elif 200001 <= income <= 1000000:
            return 'B'
        elif income >= 1000001:
            return 'A'
    except:
        pass

In [18]:
data['total_income_category'] = data['total_income'].apply(categorize_income)

**Задание 17. Выведите на экран перечень уникальных целей взятия кредита из столбца `purpose`.**

In [19]:
data['purpose'].unique()

array(['покупка жилья', 'приобретение автомобиля',
       'дополнительное образование', 'сыграть свадьбу',
       'операции с жильем', 'образование', 'на проведение свадьбы',
       'покупка жилья для семьи', 'покупка недвижимости',
       'покупка коммерческой недвижимости', 'покупка жилой недвижимости',
       'строительство собственной недвижимости', 'недвижимость',
       'строительство недвижимости', 'на покупку подержанного автомобиля',
       'на покупку своего автомобиля',
       'операции с коммерческой недвижимостью',
       'строительство жилой недвижимости', 'жилье',
       'операции со своей недвижимостью', 'автомобили',
       'заняться образованием', 'сделка с подержанным автомобилем',
       'получение образования', 'автомобиль', 'свадьба',
       'получение дополнительного образования', 'покупка своего жилья',
       'операции с недвижимостью', 'получение высшего образования',
       'свой автомобиль', 'сделка с автомобилем',
       'профильное образование', 'высшее об

**Задание 18. Создайте функцию, которая на основании данных из столбца `purpose` сформирует новый столбец `purpose_category`, в который войдут следующие категории:**

- `'операции с автомобилем'`,
- `'операции с недвижимостью'`,
- `'проведение свадьбы'`,
- `'получение образования'`.

**Например, если в столбце `purpose` находится подстрока `'на покупку автомобиля'`, то в столбце `purpose_category` должна появиться строка `'операции с автомобилем'`.**

**Используйте собственную функцию с именем `categorize_purpose()` и метод `apply()`. Изучите данные в столбце `purpose` и определите, какие подстроки помогут вам правильно определить категорию.**

In [20]:
def categorize_purpose(row):
    try:
        if 'автом' in row:
            return 'операции с автомобилем'
        elif 'жил' in row or 'недвиж' in row:
            return 'операции с недвижимостью'
        elif 'свад' in row:
            return 'проведение свадьбы'
        elif 'образов' in row:
            return 'получение образования'
    except:
        return 'нет категории'

In [21]:
data['purpose_category'] = data['purpose'].apply(categorize_purpose)

### Шаг 3. Исследуйте данные и ответьте на вопросы

#### 3.1 Есть ли зависимость между количеством детей и возвратом кредита в срок?

Заранее уточним то, что ```debt = 0``` означает, что у заёмщика нет задолжности, а ```debt = 1``` наоборот, что задолжность есть. 


На основе содержания поставленного вопроса выведем сводную таблицу методом **pivot_table()**.

Аргументы метода передадим следующие:
- *index* =```'children'``` (группируем данные по этому столцу)
- *columns* =```'debt'``` (по значениям столбца происходит группировка)
- *values* =```'education_id'``` (значения, по которым будем видеть)
- *aggfunc* =```'count'``` (функция, применяемая к значениям, то есть к *values*)

В данном случае мы передаем параметру *values* знначение столбца ```'education_id'```. Данный выбор не повлияет на результат, так как мы проводим подсчёт количества, как и указано в параметре *aggfunc* =```'count'```, и это означает, что применнение данной функции к любому столбцу приведёт к одним и тем же значениям в столбцах.   

In [22]:
# Примененим метода pivot_table() и выведем результат работы:
# Переменная data_child_debt обозначает связь между количеством детей и возвратом кредита в срок
data_child_debt = data.pivot_table(index='children', columns='debt', values='education_id', aggfunc='count')
data_child_debt

debt,0,1
children,,
0,13028.0,1063.0
1,4364.0,444.0
2,1858.0,194.0
3,303.0,27.0
4,37.0,4.0
5,9.0,NaN


Мы получили сводную таблицу, но последнее значение в столбце '1' - ```'NaN'```.

Это странно, поэтому ёще раз проверим столбец ```'debt'``` из датафрейма на наличие пропусков и выведем результат на экран. 

In [23]:
print('Наличие пропусков в столбце \'debt\' :', data['debt'].isna().sum())

Наличие пропусков в столбце 'debt' : 0


Также проверим значения в столбце ```'debt'``` при наличии 5 детей у заёмщика и выведем количество таких людей на экран:  

In [24]:
print('Количество заёмщиков, у которых 5 детей: ',data[data['children'] == 5]['debt'].count())
print('Значения в столбце \'debt\' заёмщиков, у которых 5 детей:\n', data[data['children'] == 5]['debt'])

Количество заёмщиков, у которых 5 детей:  9
Значения в столбце 'debt' заёмщиков, у которых 5 детей:
 3979     0
4397     0
7866     0
15822    0
15916    0
16211    0
20452    0
20837    0
21156    0
Name: debt, dtype: int64


Из проведенного анализа можно сказать, что последнее значение в столбце '1' - ```'NaN'``` = 0, поэтому заполним эту ячейку ```0``` и выведем повторно сводную таблицу:

In [25]:
# Заполняем ячейку со значением NaN значением 0
data_child_debt[1] = data_child_debt[1].fillna(0)

In [26]:
# Выведем полученную таблицу
data_child_debt

debt,0,1
children,,
0,13028.0,1063.0
1,4364.0,444.0
2,1858.0,194.0
3,303.0,27.0
4,37.0,4.0
5,9.0,0.0


Добавим к сводной таблице столбец ```'sum'```, с суммой столбцов ```'data_child_debt[0]'```  и  ```'data_child_debt[1]'```, и столбец под названием ```'ratio_debt'```, который будет являться отношением числа должников ```'data_child_debt[1]'``` к общему количеству заёмщиков - столбец ```'sum'```:

In [27]:
# Создадим столбец куда запишем сумму data_child_debt[0] и data_child_debt[1]
data_child_debt['sum'] = data_child_debt[0] + data_child_debt[1]

In [28]:
# Создадим столбец куда запишем отношение количества должников к сумме должников по категориям
data_child_debt['ratio_debt'] = data_child_debt[1] / data_child_debt['sum']

In [29]:
# Выведем полученную сводную таблицу
data_child_debt

debt,0,1,sum,ratio_debt
children,,,,
0,13028.0,1063.0,14091.0,0.075438
1,4364.0,444.0,4808.0,0.092346
2,1858.0,194.0,2052.0,0.094542
3,303.0,27.0,330.0,0.081818
4,37.0,4.0,41.0,0.097561
5,9.0,0.0,9.0,0.000000


Выведем сумму процентов должников с тремя и более детьми:

In [30]:
# Сумма процентов должников с тремя и более детьми
ratio_debt_sum = 0
ratio_debt_sum += data_child_debt['ratio_debt'].loc[3:6].sum()

In [31]:
# Выведем результат суммирования в процентах
print(f'Cумма процентов должников с тремя и более детьми: {ratio_debt_sum:.2%}')

Cумма процентов должников с тремя и более детьми: 17.94%


Выведем промежуточные значения, которые необходимые для подведения итогов анализа:

In [32]:
# Запишем в sum_debt сумму всех, кто взял кредит
sum_debt = data_child_debt[0].sum() + data_child_debt[1].sum()
print('Количество заёмщиков без задолжностей:',data_child_debt[0].sum())
print('Количество заёмщиков с задолжностями:',data_child_debt[1].sum())
print('Количество заёмщиков со взятыми кредитами:',sum_debt)
print(f'Процент заёмщиков, вернувших кредит: {(data_child_debt[0].sum() / sum_debt):.2%}')
print(f'Процент заёмщиков, не вернувших кредит в срок: {(data_child_debt[1].sum() / sum_debt):.2%}')

Количество заёмщиков без задолжностей: 19599.0
Количество заёмщиков с задолжностями: 1732.0
Количество заёмщиков со взятыми кредитами: 21331.0
Процент заёмщиков, вернувших кредит: 91.88%
Процент заёмщиков, не вернувших кредит в срок: 8.12%


**Вывод:** 

Из проведённых результатов можно сделать следующие выводы:

1. При наличии у заёмщика трёх и более детей, количество кредитов, которые он берёт, существенно уменьшается, как минимум в 10 раз. Таким образом, от количества детей зависит будет ли заёмщик брать кредит. 
2. Заёмщики с тремя и более детьми возвращают кредит в срок практически без задолжностей, в сумме только 17.94% заёмщиков имеют задолжности по возврату кредиту. 
3. Самое большое число, взявших кредит, среди заёмщиков, у которых нет детей - 14091.
4. Из всех заёмщиков, которые взяли кредит, только 8.12% не вернули его в срок.

Таким образом, можно утверждать, что зависимости между количеством детей и возвратом кредита в срок ***не наблюдается***, так как около 92% заёмщиком вернули кредит в срок.

#### 3.2 Есть ли зависимость между семейным положением и возвратом кредита в срок?

На основе содержания поставленного вопроса как и в прошлый раз выведем сводную таблицу методом **pivot_table()**.

Аргументы метода передадим следующие:
- *index* =```'family_status'``` (группируем данные по этому столцу)
- *columns* =```'debt'``` (по значениям столбца происходит группировка)
- *values* =```'education_id'``` (значения, по которым будем видеть)
- *aggfunc* =```'count'``` (функция, применяемая к значениям, то есть к *values*)

Как и в прошлый раз передадим параметру *values* знначение столбца ```'education_id'```. Данный выбор не повлияет на результат, так как мы проводим подсчёт количества, как и указано в параметре *aggfunc* =```'count'```, и это означает, что применнение данной функции к любому столбцу приведёт к одним и тем же значениям в столбцах.   

In [33]:
# Примененим метод pivot_table() и выведем результат работы:
# Переменная data_fam_debt обозначает связь между семейным положением и возвратом кредита в срок
data_fam_debt = data.pivot_table(index='family_status', columns='debt', values='education_id', aggfunc='count')

In [34]:
# Выведем полученный датафрейм
data_fam_debt

debt,0,1
family_status,,
Не женат / не замужем,2523,273
в разводе,1105,84
вдовец / вдова,888,63
гражданский брак,3749,385
женат / замужем,11334,927


Добавим к сводной таблице столбец ```'sum'```, с суммой столбцов ```'data_fam_debt[0]'```  и  ```'data_fam_debt[1]'```, и столбец под названием ```'ratio_debt'```, выражающийся в процентах, который будет являться отношением числа должников ```'data_child_debt[1]'``` к общему количеству заёмщиков - столбец ```'sum'```:

In [35]:
# Создадим столбец куда запишем сумму data_fam_debt[0] и data_fam_debt[1]
data_fam_debt['sum'] = data_fam_debt[0] + data_fam_debt[1]

In [36]:
# Создадим столбец куда запишем отношение количества должников к сумме должников в зависимости от семейного положения
data_fam_debt['ratio_debt'] = (data_fam_debt[1] / data_fam_debt['sum']) * 100

In [37]:
# Выведем полученный датафрейм
data_fam_debt

debt,0,1,sum,ratio_debt
family_status,,,,
Не женат / не замужем,2523,273,2796,9.763948
в разводе,1105,84,1189,7.064760
вдовец / вдова,888,63,951,6.624606
гражданский брак,3749,385,4134,9.313014
женат / замужем,11334,927,12261,7.560558


Добавим столбец `'ratio_sum'`, содержащий в себе информацию о том, сколько процентов заёмщиков с разным семейным положением взяли кредитов в зависимости от общего числа. 

In [38]:
# Запишем в sum_debt_fam сумму всех, кто взял кредит
sum_debt_fam = data_fam_debt[0].sum() + data_fam_debt[1].sum()
data_fam_debt['ratio_sum'] = (data_fam_debt['sum'] / sum_debt_fam) * 100

In [39]:
# Выведем полученный датафрейм
data_fam_debt

debt,0,1,sum,ratio_debt,ratio_sum
family_status,,,,,
Не женат / не замужем,2523,273,2796,9.763948,13.107684
в разводе,1105,84,1189,7.064760,5.574047
вдовец / вдова,888,63,951,6.624606,4.458300
гражданский брак,3749,385,4134,9.313014,19.380245
женат / замужем,11334,927,12261,7.560558,57.479724


Выведем промежуточные значения, которые необходимые для подведения итогов анализа:

In [40]:
# Запишем в sum_debt_f сумму всех, кто взял кредит
sum_debt_f = data_fam_debt[0].sum() + data_fam_debt[1].sum()
print('Количество заёмщиков без задолжностей:',data_fam_debt[0].sum())
print('Количество заёмщиков с задолжностями:',data_fam_debt[1].sum())
print('Количество заёмщиков со взятыми кредитами:',sum_debt_f)
print(f'Процент заёмщиков, вернувших кредит: {(data_fam_debt[0].sum() / sum_debt_f):.2%}')
print(f'Процент заёмщиков, не вернувших кредит в срок: {(data_fam_debt[1].sum() / sum_debt_f):.2%}')

Количество заёмщиков без задолжностей: 19599
Количество заёмщиков с задолжностями: 1732
Количество заёмщиков со взятыми кредитами: 21331
Процент заёмщиков, вернувших кредит: 91.88%
Процент заёмщиков, не вернувших кредит в срок: 8.12%


*Промежуточный вывод:*

Из результатов полученного датафрейма можно сказать следующее:
1. Наибольшее число кредитов взято заёмщиками, чьё семейное положение *женат / замужем*
2. Самое большое значение среди тех, кто не вернул кредит в срок, также у заёмщиков со статусом *женат / замужем* 
3. Наименьшее число тех, кто брал кредит и имел задолжность по нему, у *вдовцов/вдов* - всего лишь 6.62%
4. Заёмщики, состоящие в *гражданском браке* и *не женат / не замужем*, имеют одинаковые проценты должников по возврату кредита в срок с разницей в 0.45%: 9.31% и 9.76% соответственно.

**Вывод:** 

Из проведенных расчётов можно заявлять о том, что зависимость между семейным положением и возвратом кредита в срок **есть**, так как от семейного статуса зависит возьмёт ли заёмщик кредит. Также стоит отметить, что более 90% заёмщиков с разными семейными положениями вернули кредит в срок.

#### 3.3 Есть ли зависимость между уровнем дохода и возвратом кредита в срок?

На основе содержания поставленного вопроса выведем сводную таблицу методом **pivot_table()**.

Аргументы метода передадим следующие:
- *index* =```'total_income_category'``` (группируем данные по этому столцу)
- *columns* =```'debt'``` (по значениям столбца происходит группировка)
- *values* =```'education_id'``` (значения, по которым будем видеть)
- *aggfunc* =```'count'``` (функция, применяемая к значениям, то есть к *values*)

Как и в прошлый раз передадим параметру *values* знначение столбца ```'education_id'```. Данный выбор не повлияет на результат, так как мы проводим подсчёт количества, как и указано в параметре *aggfunc* =```'count'```, и это означает, что применнение данной функции к любому столбцу приведёт к одним и тем же значениям в столбцах.   

In [41]:
# Примененим метод pivot_table() и выведем результат работы:
# Переменная data_income_debt обозначает связь между категорией уровня дохода и возвратом кредита в срок
data_income_debt = data.pivot_table(index='total_income_category', columns='debt', values='education_id', aggfunc='count')

In [42]:
# Выведем полученный датафрейм
data_income_debt

debt,0,1
total_income_category,,
A,23,2
B,4660,354
C,14568,1353
D,328,21
E,20,2


Напомним диапазон зарплат, которые входят в определенную категорию:


- 0–30000 — `'E'`;
- 30001–50000 — `'D'`;
- 50001–200000 — `'C'`;
- 200001–1000000 — `'B'`;
- 1000001 и выше — `'A'`.

Добавим к сводной таблице столбец ```'sum'```, с суммой столбцов ```'data_fam_debt[0]'```  и  ```'data_fam_debt[1]'```, и столбец под названием ```'ratio_debt'```, выражающийся в процентах, который будет являться отношением числа должников ```'data_child_debt[1]'``` к общему количеству заёмщиков - столбец ```'sum'```:

In [43]:
# Создадим столбец куда запишем сумму data_fam_debt[0] и data_fam_debt[1]
data_income_debt['sum'] = data_income_debt[0] + data_income_debt[1]

In [44]:
# Создадим столбец куда запишем отношение количества должников к сумме должников в зависимости от категории дохода
data_income_debt['ratio_debt'] = (data_income_debt[1] / data_income_debt['sum']) * 100

In [45]:
# Выведем полученный датафрейм
data_income_debt

debt,0,1,sum,ratio_debt
total_income_category,,,,
A,23,2,25,8.000000
B,4660,354,5014,7.060231
C,14568,1353,15921,8.498210
D,328,21,349,6.017192
E,20,2,22,9.090909


Добавим столбец `'ratio_sum'`, содержащий в себе информацию о том, сколько процентов заёмщиков определенной категории дохода взяли кредитов в зависимости от общего числа. 

In [46]:
# Запишем в sum_debt_tin сумму всех, кто взял кредит
sum_debt_tic = data_income_debt[0].sum() + data_income_debt[1].sum()
data_income_debt['ratio_sum'] = (data_income_debt['sum'] / sum_debt_tic) * 100

In [47]:
# Выведем полученный датафрейм
data_income_debt

debt,0,1,sum,ratio_debt,ratio_sum
total_income_category,,,,,
A,23,2,25,8.000000,0.117200
B,4660,354,5014,7.060231,23.505696
C,14568,1353,15921,8.498210,74.637851
D,328,21,349,6.017192,1.636116
E,20,2,22,9.090909,0.103136


Из приведенного выше датафрейма можно увидеть, что большую часть кредитов было взято заёмщиками с уровнем ежемесячного дохода категориями `'B'` и `'C'`. Найдём общий процент этих категорий и выведем на экран: 

In [48]:
# Переменная summa_huge_ratio содержит отношение суммы заёмщиков к общему числу выданных кредитов 
summa_huge_ratio = data_income_debt['ratio_sum'][1] + data_income_debt['ratio_sum'][2]
print(f'Общий процент заёмщиков, чей процент отношения суммы заёмщиков определенной категории\nк общему числу выданных кредитов больше 20%: {summa_huge_ratio:.4}%')

Общий процент заёмщиков, чей процент отношения суммы заёмщиков определенной категории
к общему числу выданных кредитов больше 20%: 98.14%


*Промежуточный вывод:*

Из результатов полученного датафрейма можно сказать следующее:
1. Наибольшее число кредитов взято заёмщиками, чей ежемесячный доход определяется категорией `'C'`
2. Самое большое значение среди тех, кто не вернул кредит в срок, также у заёмщиков чей доход от 50 до 200 тыс. условных единиц - категория `'C'` 
3. Наименьшее число тех, кто брал кредит и имел задолжность по нему, у заёмщиков с зарплатой менее 30 тыс. услов. ед. - категория `'E'`
4. Заёмщики, с доходом категории `'D'`, имеют наименьший процент задолжностей по выплате кредита - 6.017%
5. Процент заёмщиков, взявших кредит и имевших доход от 50 тыс. до 1 млн. услов. ед. - категории `'B'` и `'C'`, составляет 98.14%. 

**Вывод:** 

Из проведённого исследования можно сказать, что зависимость между ежемесячным уровнем дохода и возвратом кредита в срок **наблюдается**. Стоит также отметить то, что в зависимости от категории дохода зависит и количество взятых кредитов. Из таблицы видно, что заёмщики с категорией дохода `E` составляют наименьшее число среди людей, взявших кредит, и составляют 0.1% от общего числа выданных кредитов. Также наблюдается следующая последовательность: чем больше ежемесячный доход, тем больше клиентов берёт кредитов. Самое большое число клиентов, взявших кредит, имеют доход от 50 тыс. до 1 млн. услов. ед. - категория дохода `'B'` и `'C'`.

#### 3.4 Как разные цели кредита влияют на его возврат в срок?

На основе содержания поставленного вопроса выведем сводную таблицу методом **pivot_table()**.

Аргументы метода передадим следующие:
- *index* =```'purpose_category'``` (группируем данные по этому столцу)
- *columns* =```'debt'``` (по значениям столбца происходит группировка)
- *values* =```'education_id'``` (значения, по которым будем видеть)
- *aggfunc* =```'count'``` (функция, применяемая к значениям, то есть к *values*)

Как и в прошлый раз передадим параметру *values* знначение столбца ```'education_id'```. Данный выбор не повлияет на результат, так как мы проводим подсчёт количества, как и указано в параметре *aggfunc* =```'count'```, и это означает, что применнение данной функции к любому столбцу приведёт к одним и тем же значениям в столбцах.   

In [49]:
# Примененим метод pivot_table() и выведем результат работы:
# Переменная data_purpose_debt обозначает зависимость возврата кредита от категории цели, для которой он был взят
data_purpose_debt = data.pivot_table(index='purpose_category', columns='debt', values='education_id', aggfunc='count')

In [50]:
# Выведем полученную сводную таблицу
data_purpose_debt

debt,0,1
purpose_category,,
операции с автомобилем,3879,400
операции с недвижимостью,9971,780
получение образования,3619,369
проведение свадьбы,2130,183


Добавим к сводной таблице столбец ```'sum'```, с суммой столбцов ```'data_purpose_debt[0]'```  и  ```'data_purpose_debt[1]'```, и столбец под названием ```'ratio'```, который будет являться отношением столбца ```'data_purpose_debt[0]'``` к столбцу ```'data_purpose_debt[1]'```:

In [51]:
# Добавляем столбец sum к сводному датафрейму
data_purpose_debt['sum'] = data_purpose_debt[0] + data_purpose_debt[1]

In [52]:
# Создадим столбец куда запишем отношение количества должников к сумме должников в зависимости от категории цели
data_purpose_debt['ratio_debt'] = (data_purpose_debt[1] / data_purpose_debt['sum']) * 100

Добавим столбец `'ratio_sum'`, содержащий в себе информацию о том, сколько процентов заёмщиков определенной категории цели взяли кредитов в зависимости от общего числа. 

In [53]:
# Запишем в sum_debt_pc сумму всех, кто взял кредит
sum_debt_pc = data_purpose_debt[0].sum() + data_purpose_debt[1].sum()
data_purpose_debt['ratio_sum'] = (data_purpose_debt['sum'] / sum_debt_pc) * 100

In [54]:
# Выведем датафрейм после добавления столбцов
data_purpose_debt

debt,0,1,sum,ratio_debt,ratio_sum
purpose_category,,,,,
операции с автомобилем,3879,400,4279,9.347978,20.060007
операции с недвижимостью,9971,780,10751,7.255139,50.400825
получение образования,3619,369,3988,9.252758,18.695795
проведение свадьбы,2130,183,2313,7.911803,10.843373


**Вывод:** 

Из проведенного анализа таблицы можно утверждать следующее:
1. Наибольшее число кредитов и задолжностей по их возврату возникает у заёмщиков, целью которых были *операции с недвижимостью*
2. Более 50% от всего числа выданных кредитов также были связаны с *операциями с недвижимостью*
3. Меньше всего кредитов было взято заёмщиками с целью *проведения свадьбы*. Причем у таких людей также меньше всего задолжностей по их возврату.
4. Менее 10% заёмщиков с разными целями имеют задолжности по кредиту.
5. Можно выделить следующий *приоритет* получения кредита для определённых целей заёмщика в зависмости от количества взятых кредитов (от наиболее к менее):
   - Недвижимость
   - Автомобиль
   - Образование
   - Свадьба
   
Можно утверждать, что возврат кредита в срок действительно **зависит** от целей заёмщика, для которых он и берется. Из приведенного датафрейма видно, для каких целей чаще берётся кредит и какой процент заёмщиков имеют задолжности по выплатам. 

#### 3.5 Приведите возможные причины появления пропусков в исходных данных.

*Ответ:* 

Можно предположить, что причиной появления пропусков в исходном датафрейме может являться сам заёмщик, который не ввёл свои данные при оформлении, или работник банка, который не уточнил эту информацию и не внёс данные о клиенте в базу. Также можно предположить, что это связано с отправкой и открытием файла в другой операционной системе или виртуальной среде; неисключено, что при отправке или сохранении файла данные могли быть неправильно распознаны программой или затереться и на их месте образовались пропуски. 

#### 3.6 Объясните, почему заполнить пропуски медианным значением — лучшее решение для количественных переменных.

*Ответ:* 

Заполнить пропуски медианным значением - это лучшее решение для количественных переменных, так как у нас появиться возможность провести сравнение этих значенияй с исходными, которые были заполнены. В нашем случае заполнение пропусков ежемесячного дохода происходило в зависимости от категории занятости заёмщика, и это очень важно при заполнении пропусков медианным значением, так как если бы было выбранно медианное значение от доходов всех заёмщиков, то у нас бы получился некорректный результат доходов, а также и исследования, так как один заёмщик может иметь доход в 20 тыс. условных единиц, а другой в 2 млн. условных единиц, а медианное значение для всех пропусков будет одним. 

### Шаг 4: общий вывод.

В результате проведённого исследования были выполнены и получены следующие результаты:
1. Обработали и заполнили пропуски соответствующим медианным значением.
    - воспользовались методами **.isna(), .fillna() и .mean()**
2. Провели изменение типа данных столбца датафрейма
    - применили метод **.astype()**
3. Обработали дубликаты данных и удалили их
    - использовали методы **.duplicated()** и **.drop_duplicates()**
4. Категоризировали исходные данные в зависимости от категорий ежемесячного дохода и категории цели кредита
5. Проанализировали, от каких критериев зависит возврат кредита в срок 

Из полученных результатов можно сделать вывод, что возврат кредита в срок **зависит** от семейного положения, уровня дохода и целей кредита. Исходя из этих зависимостей, можно сказать, что 98.14% из всех заёмщиков имеют достаточно хороший доход в месяц (от 50 тыс. до 1 млн. услов. ед.), и оттого, сколько заёмщик получает в месяц, будет решаться возьмёт ли он кредит или нет. Отметим и то, что *вдовцы/вдовы* наименее заинтересованы во взятии кредита - всего лишь 951 человек, хотя среди них и меньше всего должников - 6,62%. Также исходя из целей, для которых берётся кредит, можно сказать с какой вероятностью клиент вернёт его без задолжностей. Было показано, что более 50% всех взятых кредитов были так или иначе связаны с операциями с недвижимостью, а процент заёмщиков с задолжностями составляет всего 7,25%. Наибольшее число задолжностей имеют заёмщики, взявшие кредит для операций с автомобилем. Наличие и количество детей никак не связаны с задолжностями по кредиту - около 92% заёмщиком вернули кредит в срок.

Исходя из выявленных зависимостей, можно спрогнозировать - вернёт ли заёмщик кредит в срок или будет иметь задолжность, и для кредитного отдела банка будут наиболее перспективны заёмщики, которые вернут кредит без задолжности:
- женатые/замужние, так как они чаще других берут кредиты - 57,47% от общего числа кредитов
- с доходом категории `'B'` и `'C'`, то есть от 50 тыс. до 1 млн. услов. ед., так как такие клиенты составляют 98.14% от числа всех заёмщиков
- целью которых являются операции с недвижимостью, так как более 50% процентов клиентов берут кредит для них и выплачивают его с меньшим количеством задолжностей - 7,25%.
